Je vais reprendre chaque cellule avec le même style : **ligne par ligne**, avec la **formule mathématique** quand c'est pertinent, et l'**intuition concrète** derrière.

---

## Cellule 1 — Imports

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
```

Chaque import a un rôle précis. `numpy` gère les matrices — les SHAP values sont fondamentalement des tableaux `(N, M)`. `pandas` structure les données avec des noms de colonnes, ce qui évite l'erreur "X does not have valid feature names" qu'on a rencontrée. `scipy.cluster.hierarchy` fournit les trois fonctions du clustering hiérarchique : `linkage` construit l'arbre, `fcluster` le coupe à une hauteur donnée, `dendrogram` le trace.

```python
try:
    from mosaic_shap.explainers import TreeSHAPExplainer
    MOSAIC_AVAILABLE = True
except ImportError:
    MOSAIC_AVAILABLE = False
```

Ce bloc rend le notebook **autonome** : si le package local n'est pas installé, on continue avec les implémentations définies dans le notebook lui-même. Ça évite un crash au démarrage.

---

## Cellule 2 — Données & Modèle

```python
data = fetch_california_housing(as_frame=True)
X_full, y_full = data.data, data.target
```

`as_frame=True` retourne un DataFrame pandas (avec les noms de colonnes) plutôt qu'un array numpy. C'est important pour éviter les warnings de sklearn plus tard.

```python
N_SAMPLE = 500
idx = np.random.choice(len(X_full), N_SAMPLE, replace=False)
X = X_full.iloc[idx].reset_index(drop=True)
```

On sous-échantillonne volontairement à 500 observations. La raison est purement computationnelle : Owen et Winter appellent `model.predict()` des milliers de fois — sur 20 000 observations ce serait prohibitif.

```python
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
r2 = r2_score(y_test, model.predict(X_test))
```

Le `random_state=42` fixe la graine aléatoire pour la **reproductibilité** — deux exécutions du notebook donnent exactement les mêmes résultats. Le R² mesure la qualité du modèle : un R² de 0.80 signifie que le modèle explique 80% de la variance des prix.

---

## Cellule 3 — Rappel Shapley (Markdown théorique)

Cette cellule pose la formule de référence :

$$\phi_i = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|!(|N|-|S|-1)!}{|N|!} \bigl[v(S \cup \{i\}) - v(S)\bigr]$$

Pour la lire concrètement : on imagine qu'on ajoute les features une par une dans un ordre aléatoire. Quand vient le tour de la feature $i$, on mesure ce qu'elle apporte en plus. On répète pour tous les ordres possibles et on moyenne. Le résultat est la contribution "juste" de $i$.

La **limitation** soulignée à la fin est la motivation du notebook entier : Shapley traite toutes les features sur un pied d'égalité. Mais MedInc et HouseAge ont une relation sémantique naturelle — elles décrivent toutes les deux un profil financier. Owen et Winter permettent d'exploiter cette structure.

---

## Cellule 4 — Calcul des SHAP values (baseline)

```python
explainer_shap = shap.TreeExplainer(model)
shap_values = explainer_shap.shap_values(X)   # (N, M)
```

`TreeExplainer` exploite la structure interne des arbres pour calculer les SHAP values **exactement** en O(T·L·D²) où T = nombre d'arbres, L = nombre de feuilles, D = profondeur. C'est beaucoup plus rapide qu'une approximation par permutation.

```python
base = explainer_shap.expected_value
residual = np.abs((shap_values.sum(axis=1) + base) - pred).mean()
```

C'est la vérification de l'**axiome d'efficience** :

$$\sum_{i=1}^{M} \phi_i(x) = f(x) - \mathbb{E}[f(X)]$$

`shap_values.sum(axis=1)` donne $\sum_i \phi_i$ pour chaque observation. En ajoutant `base` (= $\mathbb{E}[f(X)]$) et en soustrayant `pred` (= $f(x)$), on doit obtenir 0. Un `residual` < 1e-5 confirme que le calcul est correct.

---

## Cellule 5 — Visualisation Shapley

```python
mean_abs_shap = np.abs(shap_values).mean(axis=0)
order = np.argsort(mean_abs_shap)[::-1]
axes[0].barh([feature_names[i] for i in order], mean_abs_shap[order], ...)
```

`np.abs(shap_values).mean(axis=0)` calcule $\frac{1}{N}\sum_n |\phi_i(x_n)|$ pour chaque feature $i$ — c'est l'**importance globale moyenne**. `np.argsort()[::-1]` trie par ordre décroissant. Le barplot horizontal permet de lire les noms de features sans rotation.

```python
shap.summary_plot(shap_values, X, plot_type='violin', show=False)
```

Le violin plot montre la **distribution** des $\phi_i$ sur toutes les observations, pas seulement leur moyenne. Un violon étalé des deux côtés de 0 indique une feature dont l'effet dépend fortement du contexte — parfois positive, parfois négative. C'est la différence entre "MedInc est importante" (barplot) et "MedInc est positive quand le revenu est élevé et négative quand il est bas" (violin).

---

## Cellule 6 — OWENExplainer

### Le constructeur

```python
all_features = [f for g in groups for f in g]
assert sorted(all_features) == list(range(self.M))
```

Cette assertion vérifie que les groupes forment une **partition complète et disjointe** de $\{0, \ldots, M-1\}$. Si une feature est oubliée ou apparaît dans deux groupes, le calcul serait mathématiquement invalide — on préfère planter immédiatement avec un message clair.

### `_v_batch` — le cœur de l'optimisation

```python
X_big = np.tile(self.background, (n_coal, 1))   # (n_coal × n_bg, M)
for k, active in enumerate(list_of_active):
    X_big[k*n_bg:(k+1)*n_bg, active] = x[active]
preds = self.model.predict(X_big)               # un seul appel !
return preds.reshape(n_coal, n_bg).mean(axis=1)
```

Plutôt que d'appeler `predict()` une fois par coalition, on empile toutes les matrices d'évaluation verticalement et on fait **un seul `predict()`**. Pour chaque coalition $S$, la marginalisation donne :

$$v(S) = \mathbb{E}[f(X) \mid X_S = x_S] \approx \frac{1}{n_{bg}} \sum_{b=1}^{n_{bg}} f(x_S, X_{\bar{S}}^{(b)})$$

Les features actives prennent la valeur de $x$, les features inactives gardent la valeur du background. Le gain est de l'ordre de 100× par rapport à des appels séparés.

### `shap_values` — l'estimateur Monte-Carlo

```python
group_order = np.random.permutation(self.K)
feature_order_by_group = [np.random.permutation(self.groups[k]) for k in range(self.K)]
```

À chaque permutation, on tire deux niveaux d'ordre aléatoire : l'ordre des groupes, puis l'ordre des features à l'intérieur de chaque groupe. La contribution marginale de la feature $i \in G_k$ est :

$$\Delta_i = v\!\left(\bigcup_{G_q \prec G_k} G_q \cup T \cup \{i\}\right) - v\!\left(\bigcup_{G_q \prec G_k} G_q \cup T\right)$$

où $T$ est la sous-coalition de features de $G_k$ déjà placées avant $i$ dans l'ordre intra-groupe. En moyennant sur toutes les permutations, on obtient la valeur d'Owen.

---

## Cellule 7 — Groupes métier

```python
GROUPS_BUSINESS = [
    [0, 1],   # Financier     (MedInc, HouseAge)
    [2, 3],   # Logement      (AveRooms, AveBedrms)
    [4, 5],   # Démographique (Population, AveOccup)
    [6, 7],   # Géographique  (Latitude, Longitude)
]
```

Ces groupes sont définis par **expertise métier** : on regroupe les features qui partagent une signification commune dans le domaine immobilier. C'est la stratégie la plus naturelle mais aussi la plus subjective.

```python
eff_err = np.abs(owen_biz.sum(axis=1) - (preds - baseline)).mean()
```

Même vérification d'efficience que pour Shapley :

$$\sum_{i=1}^{M} \phi_i^{\text{Owen}}(x) = f(x) - \mathbb{E}[f(X)]$$

Owen hérite de cette propriété de Shapley. Si `eff_err` est petit, l'implémentation est correcte.

---

## Cellule 8 — Comparaison visuelle Shapley vs Owen

```python
for j, feat_idx in enumerate(order):
    g = next(k for k, g in enumerate(GROUPS_BUSINESS) if feat_idx in g)
    bars[j].set_color(group_colors[g])
```

Chaque barre Owen est colorée selon son groupe d'appartenance. Si deux features du même groupe ont des barres de même couleur côte à côte, on voit visuellement leur "solidarité" dans la structure Owen. Ce qui est intéressant à observer : si la hauteur relative d'une feature change entre Shapley et Owen, c'est que la structure de groupe **redistribue** son importance.

---

## Cellule 9 — Scatter Shapley vs Owen

```python
ax.scatter(shap_test_arr[:, feat_idx], owen_biz[:, feat_idx], ...)
corr = np.corrcoef(shap_test_arr[:, feat_idx], owen_biz[:, feat_idx])[0, 1]
ax.text(0.05, 0.90, f"r={corr:.2f}", ...)
```

Pour chaque feature, on compare $\phi_i^{\text{Shapley}}(x_n)$ vs $\phi_i^{\text{Owen}}(x_n)$ pour chaque observation $n$. La diagonale pointillée représente l'égalité parfaite. La corrélation $r$ est l'indicateur clé :

- $r \approx 1$ → Owen = Shapley pour cette feature, le groupe n'apporte rien
- $r < 0.9$ → Owen redistribue l'importance, la structure de groupe est significative

Ces features à faible $r$ sont les plus intéressantes — elles révèlent des **effets de groupe** non capturés par Shapley standard.

---

## Cellule 10 — Fonctions de groupement automatique

### `discover_groups_from_correlation`

```python
corr = np.corrcoef(X.T)       # (M, M) corrélation entre features
dist = 1 - np.abs(corr)       # distance : features corrélées = proches
Z = linkage(pdist(X.T, metric='correlation'), method=method)
labels = fcluster(Z, t=K, criterion='maxclust')
```

On transforme la corrélation en distance ($d_{ij} = 1 - |r_{ij}|$), puis on construit un dendrogramme par linkage de Ward (qui minimise la variance intra-cluster). `fcluster` coupe l'arbre pour obtenir exactement $K$ groupes.

### `discover_groups_from_shap`

```python
corr = np.corrcoef(shap_vals.T)   # corrélation dans l'espace ES
```

Même principe mais sur les SHAP values. Différence conceptuelle importante : on regroupe les features qui ont des **effets similaires sur la prédiction**, pas des valeurs similaires dans les données. C'est généralement plus pertinent pour l'explicabilité.

---

## Cellule 11 — Calcul Owen pour les 3 stratégies

```python
for name, groups in [("Métier", GROUPS_BUSINESS), ("Corrélation", GROUPS_CORR), ("SHAP", GROUPS_SHAP)]:
    exp = OWENExplainer(model, groups, background, n_permutations=N_PERM_QUICK)
    results_owen[name] = exp.shap_values(X_test_arr)
```

On calcule Owen trois fois avec des groupes différents et on compare les rankings d'importance. Si les trois donnent le même ranking, le choix de groupement importe peu. Si ils divergent, il faudra justifier le choix retenu dans le rapport.

---

## Cellule 12 — Visualisation ranking par stratégie

```python
base_order = np.argsort(all_vals[0])[::-1]   # ordre fixé par Shapley
for ax, vals, title in zip(axes, all_vals, titles):
    ax.bar([feature_names[i] for i in base_order], vals[base_order], ...)
```

Les quatre graphiques utilisent le **même ordre de features** (celui de Shapley) pour faciliter la comparaison visuelle. Si une barre qui était en deuxième position sous Shapley passe cinquième sous Owen-SHAP, l'œil le perçoit immédiatement sans avoir à lire les chiffres.

---

## Cellule 13 — Kendall τ

```python
tau, _ = kendalltau(all_mean_abs[i], all_mean_abs[j])
```

Le **τ de Kendall** mesure la concordance entre deux rankings. Pour deux features tirées au hasard, est-ce que méthode A et méthode B s'accordent sur laquelle est la plus importante ? $\tau = 1$ signifie accord parfait, $\tau = 0$ indépendance totale.

$$\tau = \frac{\text{paires concordantes} - \text{paires discordantes}}{\binom{M}{2}}$$

La heatmap visualise toutes les paires. Les cases hors-diagonale proches de 1 indiquent que les méthodes convergent, proches de 0 qu'elles divergent significativement.

---

## Cellule 14 — WINTERExplainer

### Le constructeur — vérification de cohérence hiérarchique

```python
for kf, fg in enumerate(fine_groups):
    kc = self._feat_to_coarse[fg[0]]
    assert all(self._feat_to_coarse[f] == kc for f in fg)
```

En plus de vérifier que chaque liste est une partition complète, on vérifie que **les groupes fins ne chevauchent pas les groupes grossiers** — un groupe fin doit être entièrement contenu dans un seul groupe grossier.

### `shap_values` — trois niveaux imbriqués

```python
coarse_order = np.random.permutation(self.KC)                     # niveau 1
fine_order_by_coarse = {kc: np.random.permutation(...)}           # niveau 2
feat_order_by_fine   = {kf: np.random.permutation(...)}           # niveau 3
```

La valeur de Winter pour $i \in g_j^k \subseteq G_k$ est :

$$\phi_i^W = \sum_{Q \subseteq \mathcal{G} \setminus \{G_k\}} w_K(|Q|) \sum_{R \subseteq \mathcal{P}_{G_k} \setminus \{g_j^k\}} w_{L_k}(|R|) \sum_{T \subseteq g_j^k \setminus \{i\}} w_{|g_j^k|}(|T|) \cdot \Delta_i(Q,R,T)$$

Les trois sommes imbriquées correspondent exactement aux trois niveaux de permutation dans le code.

---

## Cellule 15 — Validation Winter plate = Owen

```python
winter_flat = WINTERExplainer(
    coarse_groups=flat_groups,
    fine_groups=flat_groups,   # même chose aux deux niveaux
    ...
)
corr_flat = np.corrcoef(wv_flat.ravel(), ov_biz_30.ravel())[0, 1]
```

C'est un **test de cohérence mathématique** : si les groupes fins sont identiques aux groupes grossiers, il n'y a qu'un seul niveau réel et Winter doit se réduire à Owen. La corrélation doit être > 0.97. Si ce n'est pas le cas, il y a un bug dans l'implémentation de l'un des deux.

---

## Cellule 16 — Hiérarchie concrète

```python
COARSE_GROUPS = [[0,1,2,3], [4,5,6,7]]   # Socio-éco vs Spatial
FINE_GROUPS   = [[0,1], [2,3], [4,5], [6,7]]
```

La structure est :

```
Niveau grossier :   [Socio-économique]        [Spatial]
                     /           \             /       \
Niveau fin :    [Financier] [Logement]  [Démo] [Géo]
                  /  \         /  \      /  \    /  \
Features :      f0   f1      f2   f3   f4  f5  f6   f7
```

Cette hiérarchie est défendable conceptuellement et sera utilisée pour interpréter si l'effet d'une feature passe plutôt par son groupe fin ou son groupe grossier.

---

## Cellule 17 — Calcul Winter

```python
winter_values = winter_explainer.shap_values(X_test_arr)
eff_err_w = np.abs(winter_values.sum(1) - (model.predict(X_test_arr) - model.predict(background).mean())).mean()
```

Même vérification d'efficience qu'Owen et Shapley. Winter hérite aussi de cette propriété :

$$\sum_{i=1}^{M} \phi_i^W(x) = f(x) - \mathbb{E}[f(X)]$$

---

## Cellule 18 — Grande figure de comparaison

```python
ax.hexbin(sv_flat, ov_flat, gridsize=30, cmap='YlOrRd', mincnt=1)
```

Le `hexbin` découpe le plan en hexagones et colore chaque hexagone selon le nombre de points qu'il contient. C'est plus lisible qu'un scatter classique quand les points se superposent — les zones rouges sont denses, les zones jaunes creuses. Sur la diagonale, les zones denses indiquent un accord entre les deux méthodes.

```python
group_contrib_owen   = np.array([np.abs(results_owen["Métier"])[:, g].mean() for g in GROUPS_BUSINESS])
group_contrib_winter = np.array([np.abs(winter_values)[:, g].mean()           for g in GROUPS_BUSINESS])
```

Ceci calcule la **contribution moyenne de chaque groupe** en sommant les importances de ses features. Comparer Owen vs Winter sur ce graphique révèle si la hiérarchie supplémentaire de Winter redistribue les contributions entre groupes.

---

## Cellule 19 — Rankings et Kendall final

```python
ranking = np.argsort(mean_abs)[::-1]
summary_data.append({"Méthode": name, **{f"Rang {r+1}": feature_names[ranking[r]] for r in range(M)}})
```

Ce tableau est le résultat le plus directement exploitable pour le rapport. Il permet de dire concrètement : "MedInc reste en tête pour les trois méthodes, mais AveOccup monte de la 6ème à la 3ème place avec Winter, ce qui suggère que son effet est amplifié par l'appartenance au groupe Spatial."

---

## Cellule 20 — `discover_two_level_hierarchy`

```python
Z = linkage(squareform(dist), method='ward')
labels_coarse = fcluster(Z, t=K_coarse, criterion='maxclust')
labels_fine   = fcluster(Z, t=K_fine,   criterion='maxclust')
```

On coupe le **même dendrogramme** à deux hauteurs différentes. Puisque les deux coupures viennent du même arbre, la cohérence hiérarchique est garantie automatiquement — un groupe fin sera toujours contenu dans le groupe grossier correspondant. Le graphique montre deux lignes horizontales de couleurs différentes sur le dendrogramme, une pour chaque niveau de coupure.

---

## Cellule 21 — Winter manuel vs Winter automatique

```python
corr_w = np.corrcoef(winter_values.ravel(), wv_auto.ravel())[0, 1]
print("→ Une corrélation < 0.90 indique que le choix de hiérarchie est significatif")
```

Si $r > 0.95$ : les deux hiérarchies sont équivalentes, le choix n'a pas grande importance. Si $r < 0.90$ : les deux hiérarchies décrivent des structures différentes. Dans ce cas il faut argumenter laquelle est la plus pertinente — la hiérarchie métier est plus interprétable, la hiérarchie automatique est plus objective.

---

## Cellule 22 — `winter_by_region`

```python
for r in unique_regions:
    wv_r = winter_exp.shap_values(X[idx_r])
    regional_means.append(np.abs(wv_r).mean(0))
heterogeneity = regional_means.std(0)     # (M,)
```

Pour chaque parcelle $r$, on calcule le vecteur $\bar{w}_r \in \mathbb{R}^M$ des importances moyennes de Winter. L'hétérogénéité de la feature $i$ est :

$$h_i = \text{std}_r(\bar{w}_{r,i})$$

Une $h_i$ élevée signifie que le rôle de la feature $i$ **change drastiquement selon les régions** — c'est le signal qu'une tessellation plus fine est nécessaire dans ces zones.

---

## Cellule 23 — Simulation HDBSCAN

```python
region_labels = hdbscan.HDBSCAN(min_cluster_size=15).fit_predict(shap_test_arr)
```

HDBSCAN cluster les observations dans l'**espace des explications** (les SHAP values) plutôt que dans l'espace des features. C'est cohérent avec la philosophie AntakIA : on veut des régions où le modèle se comporte de manière homogène, pas juste des régions où les données se ressemblent. Les points avec label `-1` sont du bruit — des observations atypiques qui n'appartiennent à aucune région.

---

## Cellule 24 — Heatmap Winter régionalisé

```python
im = ax.imshow(rm, aspect='auto', cmap='YlOrRd')   # (R, M) heatmap
ax.bar([feature_names[i] for i in order_h], heterogeneity[order_h], ...)
```

La heatmap à gauche est une matrice régions × features. Lire une **ligne** donne le profil explicatif d'une parcelle — quelles features dominent dans cette région. Lire une **colonne** montre si la feature est uniformément importante partout ou non.

Le barplot à droite classe les features par hétérogénéité décroissante. Les features à gauche sont celles dont le rôle varie le plus entre parcelles — ce sont les candidates naturelles pour affiner la segmentation dans GRANITE.

---

## Cellule 25 — Divergence Winter–Shapley

```python
divergence = winter_values - shap_test_arr   # (N, M)
mean_div = np.abs(divergence).mean(0)
```

La divergence $\delta_i = \phi_i^W - \phi_i^{\text{Shapley}}$ n'est pas une erreur, c'est un **signal**. Selon la théorie (doc `shapley_causal_vs_winter.md`), quand la hiérarchie de Winter respecte la topologie causale, cette divergence est liée aux effets de médiation. Les features à forte divergence sont des candidates pour une analyse causale approfondie.

```python
group_colors = [...]
for j, feat_idx in enumerate(order_d):
    g = next(k for k, g in enumerate(GROUPS_BUSINESS) if feat_idx in g)
    bars[j].set_color(group_colors[g])
```

La coloration par groupe permet de voir si la divergence est structurée — si toutes les features d'un même groupe ont une forte divergence, c'est que ce groupe a une dynamique interne forte non capturée par Shapley.

---

## Cellule 26 — Roadmap causale

```python
print("""
  v(S) = E[f | X_S = x_S]       ← conditionnel (actuel)
  v(S) = E[f | do(X_S = x_S)]   ← interventionnel (futur)
""")
```

C'est la transition clé vers le PSC Causal. Actuellement, la valeur caractéristique $v(S)$ utilise une **espérance conditionnelle** — on marginalise les features absentes selon leur distribution observée. Le do-calculus remplace ça par une **intervention** : on fixe les features actives indépendamment de leur distribution naturelle, ce qui élimine les effets de confounders. Winter est le pont entre les deux car sa structure hiérarchique mime naturellement cet opérateur d'intervention quand la hiérarchie reflète le DAG causal.

---

## Cellule 27 — Export final

```python
os.makedirs("figures", exist_ok=True)
print("  mosaic_shap/explainers.py  ← OWENExplainer, WINTERExplainer")
print("  mosaic_shap/grouping.py    ← discover_groups_from_correlation, ...")
```

Cette cellule a deux rôles. Elle vérifie que toutes les figures ont bien été générées (le `✅` vs `⏳`). Et elle rappelle explicitement dans quels fichiers du projet copier chaque classe — quand quelqu'un reprend le projet dans trois semaines, il sait exactement quoi faire sans relire tout le notebook.